In [119]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import t
import os

In [120]:
# Load the data
file_path = r'N:\FWET\FDCH\AdsCatal\General\personal_work_folders\plasmacatdesign\drm\ugent\ugent_drm_reaction_data_combined.parquet'
data = pd.read_parquet(file_path)

In [121]:
# select the data for which compound is CO2, CH4 or Total
data = data.loc[data['compound'].isin(['CO2', 'CH4', 'Total'])].reset_index(drop=True)

# keep only the necessary columns
data = data[
	[
		'material',
		'compound',
		'res_time_sec',
		'conv',
		'conv_sd',
        'conc_avg_corr',
        'conc_avg_corr_sd'
	]
]

# add a column "conc_init" which is the initial concentration of the compound
# CO2 and CH4 have a conc_init of 0.5 and Total has a conc_init of 1
data['conc_init'] = np.where(data['compound'] == 'Total', 1, 0.5)

# remove the row for which res_time_sec is 80 and material is 2% Cu(II)(NO3)2 @ SASOL 1.8
data_to_fit = data.loc[~((data['res_time_sec'] == 80) & (data['material'] == '2% Cu(II)(NO3)2 @ SASOL 1.8'))].reset_index(drop=True)

In [122]:
alpha_table = {
	'SASOL 1.8': 1.02,
	'2% Cu(II)(NO3)2 @ SASOL 1.8': 1.00,
	'10% Cu(II)(NO3)2 @ SASOL 1.8': 0.96,
	'2% Fe(III)Citrate @ SASOL 1.8': 1.00,
	'10% Fe(III)Citrate @ SASOL 1.8': 1.01
}

results = []

# Group data by 'material' and 'compound'
grouped = data_to_fit.groupby(['material', 'compound'])

for (material, compound), subset in grouped:
	# Get alpha value
	alpha = alpha_table[material]

	# update the reaction_rate_model with alpha
	def reaction_rate_model(res_time, k, conv_eq):
		return conv_eq - conv_eq * alpha * np.exp(-k * res_time)

	# Fit the model
	popt, pcov = curve_fit(
		f=reaction_rate_model, 
		xdata=subset['res_time_sec'],
		ydata=subset['conv'],
		p0=[0.05, 0.50],
		sigma=subset['conv_sd'],
		absolute_sigma=True
	)

	k, conv_eq = popt
	k_sd, conv_eq_sd = np.sqrt(np.diag(pcov))
	k_rsd = k_sd / k
	conv_eq_rsd = conv_eq_sd / conv_eq

	# Create prediction
	res_time_pred = np.linspace(0, 100, 201)
	fit_pred = reaction_rate_model(res_time_pred, k, conv_eq)

	for i in range(len(res_time_pred)):
		results.append({
			'material': material,
			'compound': compound,
			'res_time_sec': res_time_pred[i],
			'fit_pred': fit_pred[i],
			'conv_eq': conv_eq,
			'conv_eq_sd': conv_eq_sd,
			'conv_eq_rsd': conv_eq_rsd,
			'k': k,
			'k_sd': k_sd,
			'k_rsd': k_rsd
		})

# Convert results to DataFrame and write to CSV
fit_results = pd.DataFrame(results)

In [123]:
# left join the fit_results to the original data
data_conv_fit = fit_results.merge(data, how='left', on=['material', 'compound', 'res_time_sec'])

# write the data to a parquet file
data_conv_fit.to_parquet(os.path.join(os.path.dirname(file_path), 'ugent_drm_reaction_conv_fit.parquet'))

# write the data to a csv file
data_conv_fit.to_csv(os.path.join(os.path.dirname(file_path), 'ugent_drm_reaction_conv_fit.csv'), index=False)

C:\Users\sbossier\AppData\Local\Temp\ipykernel_20692\1402556935.py:2: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  data_conv_fit = fit_results.merge(data, how='left', on=['material', 'compound', 'res_time_sec'])


In [124]:
# Function to fit the first-order reaction rate model
def reaction_rate_model(res_time, k, mole_fraction_eq, conc_init):
	return mole_fraction_eq - (mole_fraction_eq - conc_init) * np.exp(-k * res_time)

results = []

# Group data by 'material' and 'compound'
grouped = data_to_fit.groupby(['material', 'compound'])

for (material, compound), subset in grouped:
	# Get alpha value
	alpha = alpha_table[material]

	# retrieve the initial mole fraction of the compound
	conc_init = subset['conc_init'].iloc[0]

	# Define the lambda function to fit the model with conc_init
	def reaction_rate_model_lambda(res_time, k, mole_fraction_eq):
		return reaction_rate_model(res_time, k, mole_fraction_eq, conc_init)

	# Fit the model
	popt, pcov = curve_fit(
		f=reaction_rate_model_lambda, 
		xdata=subset['res_time_sec'],
		ydata=subset['conc_avg_corr'],
		p0=[0.05, 0.20],
		sigma=subset['conc_avg_corr_sd'],
		absolute_sigma=True
	)

	k, mole_fraction_eq = popt
	k_sd, mole_fraction_eq_sd = np.sqrt(np.diag(pcov))
	k_rsd = k_sd / k
	mole_fraction_eq_rsd = mole_fraction_eq_sd / mole_fraction_eq

	conv_eq = 1 - (alpha * mole_fraction_eq) / conc_init
	conv_eq_sd = mole_fraction_eq_sd / conc_init

	f_k_form = k * mole_fraction_eq
	f_k_form_rsd = np.sqrt(k_rsd**2 + mole_fraction_eq_rsd**2)
	f_k_form_sd = f_k_form_rsd * f_k_form

	k_loss = (1 - mole_fraction_eq) * k
	k_loss_rsd = np.sqrt(
		k_rsd**2 + (mole_fraction_eq_sd / (1 - mole_fraction_eq))**2
	)
	k_loss_sd = k_loss_rsd * k_loss

	# Create prediction and confidence intervals
	res_time_pred = np.linspace(0, 100, 201)
	fit_pred = reaction_rate_model(res_time_pred, k, mole_fraction_eq, conc_init)
	conv_pred = 1 - (alpha * fit_pred) / conc_init

	for i in range(len(res_time_pred)):
		results.append({
			'material': material,
			'compound': compound,
			'mole_fraction_eq': mole_fraction_eq,
			'mole_fraction_eq_sd': mole_fraction_eq_sd,
			'mole_fraction_eq_rsd': mole_fraction_eq_rsd,
			'k': k,
			'k_sd': k_sd,
			'k_rsd': k_rsd,
			'f_k_form': f_k_form,
			'f_k_form_sd': f_k_form_sd,
			'f_k_form_rsd': f_k_form_rsd,
			'k_loss': k_loss,
			'k_loss_sd': k_loss_sd,
			'k_loss_rsd': k_loss_rsd,
			'conv_eq': conv_eq,
			'conv_eq_sd': conv_eq_sd,
			'res_time_sec': res_time_pred[i],
			'mole_fraction_pred': fit_pred[i],
			'conv_pred': conv_pred[i]
		})

# Convert results to DataFrame and write to CSV
fit_results = pd.DataFrame(results)

In [125]:
# left join the fit_results to the original data
data_mole_fit = fit_results.merge(data, how='left', on=['material', 'compound', 'res_time_sec'])

# write the data to a parquet file
data_mole_fit.to_parquet(os.path.join(os.path.dirname(file_path), 'ugent_drm_reaction_mole_frac_fit.parquet'))

# write the data to a csv file
data_mole_fit.to_csv(os.path.join(os.path.dirname(file_path), 'ugent_drm_reaction_mole_frac_fit.csv'), index=False)

C:\Users\sbossier\AppData\Local\Temp\ipykernel_20692\430665795.py:2: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  data_mole_fit = fit_results.merge(data, how='left', on=['material', 'compound', 'res_time_sec'])
